In [49]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm catboost xlsxwriter


In [50]:
import sys
!{sys.executable} -m pip install -q imblearn

In [51]:
from imblearn.over_sampling import SMOTE

In [52]:
import os
import re
import time
import json
import warnings
from pathlib import Path
from typing import Dict, List

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE


# =============================================================================
# CONFIG
# =============================================================================
#
# OBJECTIF : tester si les 5 modèles de référence du projet (feature sets
# figés, hyperparamètres standards) tiennent leurs métriques quand on
# déplace TEST_DATE de 2024 à 2022 — ce qui inclut le régime 2022-2024
# dans le test set (resserrement Fed, bear market, suffisamment d'épisodes
# STRESS pour évaluer ce régime correctement).
#
# Walk-forward adaptatif (folds équilibrés CALM/STRESS) sur 2022-2026,
# exactement comme VIX_BALANCED_FOLDS_NO_TRENDS, mais :
#   - PAS de sweep Phase 1 (feature sets de référence fixés en dur ci-dessous)
#   - PAS de génération d'interactions
#   - SMOTE uniquement sur CALM et STRESS (comme les modèles de référence)

OUTPUT_DIR = Path("/content/outputs_v22_reference_test2022")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START = "2000-01-01"

# TEST_DATE décalé à 2022 pour inclure la période de stress 2022-2024 dans le test
TEST_DATE = "2022-01-01"

YF_CHUNK_SIZE = 40
SLEEP_BETWEEN_CHUNKS = 1.0
MIN_COLUMN_COVERAGE = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365
VIX_FLAT_PCT_THRESHOLD = 0.0
ROLLING_QUANTILE_WINDOW = 504

# Walk-forward adaptatif
N_WALKFORWARD_FOLDS = 4
MIN_OBS_PER_REGIME_PER_FOLD = 15
WALKFORWARD_TEST_START = "2022-01-01"

np.random.seed(RANDOM_STATE)

FRED_API_KEY = os.getenv("FRED_API_KEY")
if FRED_API_KEY:
    os.environ["FRED_API_KEY"] = FRED_API_KEY

# =============================================================================
# FEATURE SETS DE RÉFÉRENCE (figés — issus des CSV de référence du projet)
# Aucun sweep de sélection n'est effectué ici : ces features sont celles
# qui ont produit les métriques de référence connues.
# =============================================================================

REFERENCE_MODELS = {
    "CALM": {
        "algo": "RandomForest",
        "features": [
            "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d",
            "T5Y5Y_Inflation_Forward_ret_1d", "KO_CocaCola_zscore_60d",
            "SP500_Price_zscore_60d", "DAX_Germany_ret_20d",
            "BA_Boeing_ret_1d", "T10Y_Inflation_Expectation_ret_1d",
            "CMCSA_Comcast_ret_20d", "GOOGL_Google_ret_5d",
            "KO_CocaCola_ret_20d", "CMCSA_Comcast_zscore_60d",
            "spx_drawdown_252d", "Nikkei_Japan_zscore_60d",
        ],
        "smote": False,
        "ref_metrics": {"F1": 0.667, "AUC": 0.615, "Precision": 0.627, "Recall": 0.712},
    },
    "NORMAL": {
        "algo": "GradientBoosting",
        "features": [
            "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d",
            "T5Y5Y_Inflation_Forward_ret_1d", "KO_CocaCola_zscore_60d",
            "SP500_Price_zscore_60d", "DAX_Germany_ret_20d",
            "BA_Boeing_ret_1d", "T10Y_Inflation_Expectation_ret_1d",
            "CMCSA_Comcast_ret_20d", "GOOGL_Google_ret_5d",
            "KO_CocaCola_ret_20d", "CMCSA_Comcast_zscore_60d",
            "spx_drawdown_252d", "Nikkei_Japan_zscore_60d",
        ],
        "smote": False,
        "ref_metrics": {"F1": 0.564, "AUC": 0.608, "Precision": 0.591, "Recall": 0.540},
    },
    "STRESS": {
        "algo": "XGBoost",
        "features": [
            "AAPL_Apple_ret_5d", "GE_GeneralElectric_ret_5d", "ABT_ret_5d",
            "SBUX_Starbucks_ret_5d", "ADBE_Adobe_ret_5d", "QCOM_Qualcomm_ret_5d",
            "QQQ_ret_5d", "V_Visa_ret_5d", "vix_vs_ma20",
            "ABT_ret_20d", "ABT_ret_1d", "VXN_NASDAQ_Vol_ret_5d",
            "MSFT_Microsoft_ret_5d", "DOW_Price_ret_5d", "KO_CocaCola_zscore_60d",
            "INTC_Intel_ret_5d", "TXN_ret_5d", "KO_CocaCola_ret_5d",
            "VXN_NASDAQ_Vol_ret_20d",
        ],
        "smote": True,
        "ref_metrics": {"F1": 0.470, "AUC": 0.613, "Precision": 0.429, "Recall": 0.519},
    },
    "GLOBAL": {
        "algo": "RandomForest",
        "features": [
            "spx_drawdown_252d", "SP500_Price_zscore_60d", "VIX_Price_zscore_60d",
            "QQQ_zscore_60d", "vix_level", "VXN_NASDAQ_Vol_zscore_60d",
            "SP500_Price_ret_5d", "VIX_Price_vol_20d",
            "NASDAQ_Price_vol_20d", "VXN_NASDAQ_Vol_vol_20d",
        ],
        "smote": True,
        "ref_metrics": {"F1": 0.545, "AUC": 0.604, "Precision": 0.535, "Recall": 0.556},
    },
}


In [53]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [54]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [55]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [56]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [57]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [58]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [59]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [60]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [61]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [62]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [63]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [64]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [65]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [66]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40
[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40
[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [67]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [68]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6734, 1180), features: 985


In [69]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [70]:
def get_model_for_regime(regime_cfg: dict):
    """Retourne l'estimateur configuré pour le régime donné,
    avec les hyperparamètres standards utilisés dans les modèles de référence."""
    algo = regime_cfg["algo"]
    if algo == "RandomForest":
        return RandomForestClassifier(
            n_estimators=150, max_depth=5, min_samples_leaf=10,
            random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
        )
    elif algo == "GradientBoosting":
        return GradientBoostingClassifier(
            n_estimators=125, learning_rate=0.05, max_depth=3,
            min_samples_leaf=10, random_state=RANDOM_STATE
        )
    elif algo == "XGBoost":
        return XGBClassifier(
            n_estimators=125, learning_rate=0.05, max_depth=3,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
        )
    elif algo == "LightGBM":
        return LGBMClassifier(
            n_estimators=125, learning_rate=0.05, num_leaves=15,
            max_depth=5, random_state=RANDOM_STATE, verbose=-1,
            class_weight="balanced"
        )
    elif algo == "LogisticRegression":
        return LogisticRegression(
            C=0.1, penalty="l2", max_iter=2000,
            random_state=RANDOM_STATE, class_weight="balanced"
        )
    else:
        raise ValueError(f"Algo inconnu : {algo}")


In [71]:
def build_balanced_walkforward_folds(df_post_features: pd.DataFrame,
                                      n_folds: int = 4,
                                      min_obs_per_regime: int = 15,
                                      test_start: str = "2022-01-01") -> list:
    """Folds walk-forward à durée variable garantissant min_obs_per_regime
    jours CALM et STRESS dans chaque fold de test. Identique à
    VIX_BALANCED_FOLDS_NO_TRENDS — les régimes sont calculés en mode
    'fixed' avec seuils sur le train (< test_start), indépendamment de
    l'horizon (VIX_Regime ne dépend pas de horizon_days)."""

    target_builder_for_folds = TargetBuilder(
        q_low=0.33, q_high=0.67, mode="fixed",
        rolling_window=ROLLING_QUANTILE_WINDOW, horizon_days=1
    )
    df_regime = target_builder_for_folds.build(df_post_features, train_end=test_start)
    regime_series = df_regime["VIX_Regime"].sort_index()
    test_zone = regime_series.loc[regime_series.index >= pd.Timestamp(test_start)]

    if test_zone.empty:
        raise ValueError(f"Aucune donnée après test_start={test_start}.")

    all_test_dates = test_zone.index.sort_values()
    folds = []
    current_train_end = pd.Timestamp(test_start)
    cursor_idx = 0

    for fold_num in range(1, n_folds + 1):
        if cursor_idx >= len(all_test_dates):
            print(f"[WARN] Plus de données pour le fold {fold_num}. {len(folds)} folds créés.")
            break

        fold_start_idx = cursor_idx
        calm_count = stress_count = 0
        end_idx = fold_start_idx

        while end_idx < len(all_test_dates):
            r = regime_series.loc[all_test_dates[end_idx]]
            if r == "CALM":   calm_count += 1
            elif r == "STRESS": stress_count += 1
            end_idx += 1
            if calm_count >= min_obs_per_regime and stress_count >= min_obs_per_regime:
                break

        if calm_count < min_obs_per_regime or stress_count < min_obs_per_regime:
            print(f"[WARN] Fold {fold_num}: minimum non atteint "
                  f"(CALM={calm_count}, STRESS={stress_count}). Fold partiel créé.")

        fold_test_start = all_test_dates[fold_start_idx]
        fold_test_end = (all_test_dates[end_idx]
                         if end_idx < len(all_test_dates)
                         else all_test_dates[-1] + pd.Timedelta(days=1))

        folds.append({
            "train_end": current_train_end.strftime("%Y-%m-%d"),
            "test_start": fold_test_start.strftime("%Y-%m-%d"),
            "test_end": fold_test_end.strftime("%Y-%m-%d"),
            "n_calm": calm_count,
            "n_stress": stress_count,
            "n_total_days": end_idx - fold_start_idx,
        })
        print(f"[FOLD {fold_num}] {fold_test_start.date()} -> {fold_test_end.date()} "
              f"({end_idx - fold_start_idx} jours, CALM={calm_count}, STRESS={stress_count})")

        current_train_end = fold_test_end
        cursor_idx = end_idx

    return folds


WALKFORWARD_FOLDS = build_balanced_walkforward_folds(
    df_post_features,
    n_folds=N_WALKFORWARD_FOLDS,
    min_obs_per_regime=MIN_OBS_PER_REGIME_PER_FOLD,
    test_start=WALKFORWARD_TEST_START,
)

folds_df = pd.DataFrame(WALKFORWARD_FOLDS)
folds_df.to_csv(OUTPUT_DIR / "balanced_folds_2022.csv", index=False)
print(f"\n[SAVE] balanced_folds_2022.csv ({len(WALKFORWARD_FOLDS)} folds)")
display(folds_df)


[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 14.81, NORMAL [14.81-21.10), STRESS >= 21.10  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6733
[TARGET] Remaining non-flat rows: 6477
VIX_Regime
NORMAL    2369
STRESS    2115
CALM      1993
Name: count, dtype: int64
[FOLD 1] 2022-01-03 -> 2023-06-27 (370 jours, CALM=15, STRESS=235)
[FOLD 2] 2023-06-27 -> 2024-11-05 (341 jours, CALM=185, STRESS=15)
[FOLD 3] 2024-11-05 -> 2025-03-31 (97 jours, CALM=20, STRESS=15)
[FOLD 4] 2025-03-31 -> 2026-01-05 (192 jours, CALM=15, STRESS=38)

[SAVE] balanced_folds_2022.csv (4 folds)


,train_end,test_start,test_end,n_calm,n_stress,n_total_days
0,2022-01-01,2022-01-03,2023-06-27,15,235,370
1,2023-06-27,2023-06-27,2024-11-05,185,15,341
2,2024-11-05,2024-11-05,2025-03-31,20,15,97
3,2025-03-31,2025-03-31,2026-01-05,15,38,192


In [72]:
# =============================================================================
# ENTRAÎNEMENT DES MODÈLES DE RÉFÉRENCE — walk-forward adaptatif
#
# Pour chaque régime (CALM/NORMAL/STRESS/GLOBAL) × chaque fold walk-forward :
#   - Feature set figé (issu de REFERENCE_MODELS, pas de sélection dynamique)
#   - Algorithme et SMOTE comme dans le modèle de référence du projet
#   - Entraînement direct (pas de GridSearchCV — hyperparamètres figés aussi)
#   - Évaluation sur le test set de ce fold
#
# Pas de sweep Phase 1 : l'objectif est de tester si les mêmes modèles,
# sur les mêmes features, tiennent leurs métriques sur une fenêtre de test
# plus large (2022-2026 au lieu de 2024-2026), en particulier pour STRESS
# qui manquait d'épisodes de test dans le run précédent.
# =============================================================================

print("="*80)
print("ENTRAÎNEMENT DES MODÈLES DE RÉFÉRENCE (TEST_DATE=2022, walk-forward adaptatif)")
print("="*80)

all_rows = []
model_number = 0

for regime, cfg in REFERENCE_MODELS.items():
    features = cfg["features"]
    algo = cfg["algo"]
    use_smote = cfg["smote"]

    print(f"\n{'='*60}")
    print(f"RÉGIME : {regime} | Algo : {algo} | SMOTE : {use_smote}")
    print(f"Features ({len(features)}) : {features}")
    print(f"{'='*60}")

    # Construire le df complet pour ce régime (target + régime)
    target_builder = TargetBuilder(
        q_low=0.33, q_high=0.67, mode="fixed",
        rolling_window=ROLLING_QUANTILE_WINDOW, horizon_days=1
    )
    df_full = target_builder.build(df_post_features, train_end=TEST_DATE)

    # Vérification AVANT le replace (ordre corrigé) : certaines features de
    # référence peuvent avoir un nom légèrement différent dans le dataset
    # courant (ex: "KO_CocaCola_zscore_60d" au lieu de "KO_zscore_60d").
    # On tente une correspondance souple : si la feature exacte n'est pas
    # trouvée, on cherche une colonne du dataset dont le nom CONTIENT le
    # nom de la feature de référence (ou vice-versa).
    def find_best_match(feat_name: str, available_cols: list) -> str:
        """Retourne la meilleure correspondance dans available_cols, ou None.
        Stratégie en 3 niveaux :
          1. Correspondance exacte (priorité absolue)
          2. Même suffixe de transformation (ex: _zscore_60d, _ret_5d) ET
             même ticker de base (les N premiers caractères avant le premier _)
          3. Aucune correspondance -> None
        La correspondance partielle naïve (col in feat ou feat in col) est
        volontairement évitée car elle retourne des faux positifs comme
        "T" pour "T5Y5Y_Inflation_Forward_ret_1d".
        """
        if feat_name in available_cols:
            return feat_name

        # Extraire le suffixe de transformation (tout ce qui vient après le
        # premier segment ticker, ex: "_zscore_60d", "_ret_5d")
        parts = feat_name.split("_")
        if len(parts) < 2:
            return None

        ticker_base = parts[0]  # ex: "KO", "BA", "CMCSA", "T5Y5Y"

        # Chercher les suffixes possibles : on teste progressivement des
        # suffixes de plus en plus courts depuis la fin du nom de feature
        # (pour gérer les noms composites comme "T5Y5Y_Inflation_Forward_ret_1d")
        for suffix_start in range(1, len(parts)):
            suffix = "_" + "_".join(parts[suffix_start:])  # ex: "_ret_1d"
            for col in available_cols:
                if col.startswith(ticker_base + "_") and col.endswith(suffix):
                    return col

        return None

    all_cols = list(df_full.columns)
    features_run = []
    for feat in features:
        match = find_best_match(feat, all_cols)
        if match:
            if match != feat:
                print(f"[MATCH] {feat} -> {match}")
            features_run.append(match)
        else:
            print(f"[WARN] {regime}: feature introuvable (même par correspondance partielle) : {feat}")

    if not features_run:
        print(f"[WARN] {regime}: aucune feature disponible. Skip.")
        continue

    print(f"[OK] {len(features_run)}/{len(features)} features disponibles "
          f"({'exactes' if len(features_run)==len(features) else 'avec correspondance partielle pour certaines'})")

    # Replace inf SEULEMENT sur les features disponibles (après la vérification)
    df_full[features_run] = df_full[features_run].replace([np.inf, -np.inf], np.nan)

    if len(features_run) == 0:
        print(f"[WARN] {regime}: aucune feature disponible. Skip.")
        continue

    for fold_idx, fold in enumerate(WALKFORWARD_FOLDS, 1):
        fold_train_end = pd.Timestamp(fold["train_end"])
        fold_test_start = pd.Timestamp(fold["test_start"])
        fold_test_end = pd.Timestamp(fold["test_end"])

        df_train_fold = df_full.loc[df_full.index < fold_train_end].copy()
        df_test_fold = df_full.loc[
            (df_full.index >= fold_test_start) &
            (df_full.index < fold_test_end)
        ].copy()

        # Masque du régime dans ce fold
        if regime == "GLOBAL":
            train_mask = pd.Series(True, index=df_train_fold.index)
            test_mask = pd.Series(True, index=df_test_fold.index)
        else:
            train_mask = df_train_fold["VIX_Regime"] == regime
            test_mask = df_test_fold["VIX_Regime"] == regime

        df_train_regime = df_train_fold.loc[train_mask]
        df_test_regime = df_test_fold.loc[test_mask]

        # Vérifications minimales
        if len(df_train_regime) < 30:
            print(f"[WARN] {regime} fold{fold_idx}: train trop petit ({len(df_train_regime)}). Skip.")
            continue
        if len(df_test_regime) < 5:
            print(f"[WARN] {regime} fold{fold_idx}: test trop petit ({len(df_test_regime)}). Skip.")
            continue

        y_train = df_train_regime["VIX_Direction"].values
        y_test = df_test_regime["VIX_Direction"].values

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"[WARN] {regime} fold{fold_idx}: une seule classe. Skip.")
            continue

        # Nettoyage + standardisation
        cleaner = TrainFittedCleaner()
        X_train_clean = cleaner.fit_transform(df_train_regime[features_run])
        X_test_clean = cleaner.transform(df_test_regime[features_run])

        scaler = StandardScaler()
        X_train = pd.DataFrame(
            scaler.fit_transform(X_train_clean),
            columns=features_run, index=df_train_regime.index
        )
        X_test = pd.DataFrame(
            scaler.transform(X_test_clean),
            columns=features_run, index=df_test_regime.index
        )

        # SMOTE si requis par le modèle de référence
        if use_smote:
            unique, counts = np.unique(y_train, return_counts=True)
            if len(unique) > 1 and min(counts) > 1:
                sm = SMOTE(random_state=RANDOM_STATE)
                X_train_arr, y_train_smote = sm.fit_resample(X_train.values, y_train)
                X_train = pd.DataFrame(X_train_arr, columns=features_run)
            else:
                y_train_smote = y_train
        else:
            y_train_smote = y_train

        # Entraînement (pas de GridSearchCV — hyperparamètres figés)
        model = get_model_for_regime(cfg)
        try:
            model.fit(X_train.values, y_train_smote)
        except Exception as e:
            print(f"[WARN] {regime} fold{fold_idx}: entraînement échoué ({e}). Skip.")
            continue

        # Prédictions et métriques
        y_pred = model.predict(X_test.values)
        y_proba = (model.predict_proba(X_test.values)[:, 1]
                   if hasattr(model, "predict_proba") else y_pred.astype(float))

        try:
            auc = roc_auc_score(y_test, y_proba)
        except Exception:
            auc = np.nan

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

        model_number += 1
        all_rows.append({
            "Model_Number": model_number,
            "VIX_Regime": regime,
            "Algo": algo,
            "N_Features": len(features_run),
            "N_Features_Available": len(features_run),
            "N_Features_Reference": len(features),
            "SMOTE_Used": use_smote,
            "Fold": fold_idx,
            "Fold_Train_End": fold["train_end"],
            "Fold_Test_Start": fold["test_start"],
            "Fold_Test_End": fold["test_end"],
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "AUC": auc,
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "Actual_0": int(tn + fp),
            "Actual_1": int(tp + fn),
            "Train_N": len(df_train_regime),
            "Test_N": len(df_test_regime),
        })

        print(f"  Fold {fold_idx} | train_n={len(df_train_regime)} test_n={len(df_test_regime)} "
              f"-> AUC={auc:.4f} F1={f1_score(y_test, y_pred, zero_division=0):.4f} "
              f"P={precision_score(y_test, y_pred, zero_division=0):.3f} "
              f"R={recall_score(y_test, y_pred, zero_division=0):.3f}")

results_df = pd.DataFrame(all_rows)
results_df.to_csv(OUTPUT_DIR / "reference_models_walkforward_results.csv", index=False)
print(f"\n[SAVE] reference_models_walkforward_results.csv ({len(results_df)} lignes)")


ENTRAÎNEMENT DES MODÈLES DE RÉFÉRENCE (TEST_DATE=2022, walk-forward adaptatif)

RÉGIME : CALM | Algo : RandomForest | SMOTE : False
Features (14) : ['DAX_Germany_zscore_60d', 'DOW_Price_zscore_60d', 'T5Y5Y_Inflation_Forward_ret_1d', 'KO_CocaCola_zscore_60d', 'SP500_Price_zscore_60d', 'DAX_Germany_ret_20d', 'BA_Boeing_ret_1d', 'T10Y_Inflation_Expectation_ret_1d', 'CMCSA_Comcast_ret_20d', 'GOOGL_Google_ret_5d', 'KO_CocaCola_ret_20d', 'CMCSA_Comcast_zscore_60d', 'spx_drawdown_252d', 'Nikkei_Japan_zscore_60d']
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 14.81, NORMAL [14.81-21.10), STRESS >= 21.10  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6733
[TARGET] Remaining non-flat rows: 6477
VIX_Regime
NORMAL    2369
STRESS    2115
CALM      1993
Name: count, dtype: int64
[WARN] CALM: feature introuvable (même par correspondance partielle) : T5Y5Y_Inflation_Forward_ret_1d
[MATCH] KO_CocaCola_zsco

In [73]:
# =============================================================================
# COMPARAISON DIRECTE AUX MÉTRIQUES DE RÉFÉRENCE
# =============================================================================

print("="*80)
print("SYNTHÈSE : métriques moyennées sur les folds walk-forward")
print("(comparées aux métriques de référence — test set 2024-2026, split fixe)")
print("="*80)

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
summary_rows = []

for regime, cfg in REFERENCE_MODELS.items():
    ref = cfg["ref_metrics"]
    sub = results_df[results_df["VIX_Regime"] == regime]

    if sub.empty:
        print(f"\n{regime}: aucun résultat disponible.")
        continue

    n_folds_ok = sub["Fold"].nunique()
    means = sub[metric_cols].mean()
    stds  = sub[metric_cols].std()

    f1_gap  = means["F1"]  - ref["F1"]
    auc_gap = means["AUC"] - ref["AUC"]

    if f1_gap > 0.01 and auc_gap > 0.005:
        verdict = "✓  AMELIORATION"
    elif f1_gap < -0.02 or auc_gap < -0.01:
        verdict = "✗  REGRESSION"
    else:
        verdict = "≈  STABLE / MARGINAL"

    stability = "STABLE" if stds["AUC"] < 0.08 else "INSTABLE"

    summary_rows.append({
        "VIX_Regime": regime,
        "Algo_Ref": cfg["algo"],
        "N_Folds": n_folds_ok,
        "AUC_mean": round(means["AUC"], 4),
        "AUC_std":  round(stds["AUC"],  4),
        "F1_mean":  round(means["F1"],  4),
        "F1_std":   round(stds["F1"],   4),
        "Prec_mean":round(means["Precision"], 3),
        "Rec_mean": round(means["Recall"],    3),
        "AUC_ref":  ref["AUC"],
        "F1_ref":   ref["F1"],
        "AUC_gap":  round(auc_gap, 4),
        "F1_gap":   round(f1_gap,  4),
        "Verdict":  verdict,
        "Stability":stability,
    })

    print(f"""
┌─ {regime} ({cfg["algo"]}, {n_folds_ok}/{N_WALKFORWARD_FOLDS} folds)
│  AUC  : {means["AUC"]:.4f} ±{stds["AUC"]:.4f}  vs ref {ref["AUC"]}  gap {auc_gap:+.4f}
│  F1   : {means["F1"]:.4f} ±{stds["F1"]:.4f}  vs ref {ref["F1"]}  gap {f1_gap:+.4f}
│  P/R  : {means["Precision"]:.3f}/{means["Recall"]:.3f}  vs ref {ref["Precision"]}/{ref["Recall"]}
│  [{stability}] [{verdict}]
└─""")

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("TABLEAU FINAL")
print("="*80)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / "reference_models_comparison_summary.csv", index=False)
print(f"[SAVE] reference_models_comparison_summary.csv")


SYNTHÈSE : métriques moyennées sur les folds walk-forward
(comparées aux métriques de référence — test set 2024-2026, split fixe)

┌─ CALM (RandomForest, 4/4 folds)
│  AUC  : 0.5636 ±0.1056  vs ref 0.615  gap -0.0514
│  F1   : 0.5865 ±0.1640  vs ref 0.667  gap -0.0805
│  P/R  : 0.608/0.570  vs ref 0.627/0.712
│  [INSTABLE] [✗  REGRESSION]
└─

┌─ NORMAL (GradientBoosting, 4/4 folds)
│  AUC  : 0.5867 ±0.0510  vs ref 0.608  gap -0.0213
│  F1   : 0.4486 ±0.1308  vs ref 0.564  gap -0.1154
│  P/R  : 0.523/0.411  vs ref 0.591/0.54
│  [STABLE] [✗  REGRESSION]
└─

┌─ STRESS (XGBoost, 4/4 folds)
│  AUC  : 0.4812 ±0.1181  vs ref 0.613  gap -0.1318
│  F1   : 0.4048 ±0.0466  vs ref 0.47  gap -0.0652
│  P/R  : 0.343/0.510  vs ref 0.429/0.519
│  [INSTABLE] [✗  REGRESSION]
└─

┌─ GLOBAL (RandomForest, 4/4 folds)
│  AUC  : 0.5998 ±0.0306  vs ref 0.604  gap -0.0042
│  F1   : 0.4879 ±0.0926  vs ref 0.545  gap -0.0571
│  P/R  : 0.511/0.475  vs ref 0.535/0.556
│  [STABLE] [✗  REGRESSION]
└─

TABLEAU FINAL


,VIX_Regime,Algo_Ref,N_Folds,AUC_mean,AUC_std,F1_mean,F1_std,Prec_mean,Rec_mean,AUC_ref,F1_ref,AUC_gap,F1_gap,Verdict,Stability
0,CALM,RandomForest,4,0.5636,0.1056,0.5865,0.1640,0.608,0.570,0.615,0.667,-0.0514,-0.0805,✗ REGRESSION,INSTABLE
1,NORMAL,GradientBoosting,4,0.5867,0.0510,0.4486,0.1308,0.523,0.411,0.608,0.564,-0.0213,-0.1154,✗ REGRESSION,STABLE
2,STRESS,XGBoost,4,0.4812,0.1181,0.4048,0.0466,0.343,0.510,0.613,0.470,-0.1318,-0.0652,✗ REGRESSION,INSTABLE
3,GLOBAL,RandomForest,4,0.5998,0.0306,0.4879,0.0926,0.511,0.475,0.604,0.545,-0.0042,-0.0571,✗ REGRESSION,STABLE


[SAVE] reference_models_comparison_summary.csv


In [74]:
# =============================================================================
# EXCEL FINAL + VERDICT INTERPRÉTATIF
# =============================================================================

excel_path = OUTPUT_DIR / "vix_reference_test2022_report.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    folds_df.to_excel(writer, sheet_name="Balanced_Folds", index=False)
    results_df.to_excel(writer, sheet_name="All_Results_By_Fold", index=False)
    summary_df.to_excel(writer, sheet_name="Summary_vs_Reference", index=False)

print(f"[SAVE] {excel_path}")
print()
print("="*80)
print("VERDICT INTERPRÉTATIF")
print("="*80)
print("""
Ce notebook répond à la question : les modèles de référence (feature sets et
algos figés, appris en dehors de ce run) tiennent-ils leurs métriques quand
on les évalue sur une fenêtre plus large (2022-2026 au lieu de 2024-2026) ?

LECTURE DES RÉSULTATS :
  ✓ AMELIORATION ou ≈ STABLE + STABLE = modèle robuste, les métriques se
    maintiennent sur un test set différent (2022-2024 inclus) -> on peut
    avoir davantage confiance dans la transférabilité de ces modèles.

  ✗ REGRESSION + INSTABLE = les métriques sont inférieures ET/OU très
    variables d'un fold à l'autre -> signal que le modèle avait partiellement
    overfit la période 2024-2026, ou que les patterns changent entre 2022
    et 2026 de façon que ce modèle ne capture pas.

NOTE MÉTHODOLOGIQUE : une amélioration sur 2022-2026 ne prouve pas que
le modèle va bien performer en production — mais une régression claire
serait un signal d'alerte sérieux sur la robustesse de la référence.
""")


[SAVE] /content/outputs_v22_reference_test2022/vix_reference_test2022_report.xlsx

VERDICT INTERPRÉTATIF

Ce notebook répond à la question : les modèles de référence (feature sets et
algos figés, appris en dehors de ce run) tiennent-ils leurs métriques quand
on les évalue sur une fenêtre plus large (2022-2026 au lieu de 2024-2026) ?

LECTURE DES RÉSULTATS :
  ✓ AMELIORATION ou ≈ STABLE + STABLE = modèle robuste, les métriques se
    maintiennent sur un test set différent (2022-2024 inclus) -> on peut
    avoir davantage confiance dans la transférabilité de ces modèles.

  ✗ REGRESSION + INSTABLE = les métriques sont inférieures ET/OU très
    variables d'un fold à l'autre -> signal que le modèle avait partiellement
    overfit la période 2024-2026, ou que les patterns changent entre 2022
    et 2026 de façon que ce modèle ne capture pas.

NOTE MÉTHODOLOGIQUE : une amélioration sur 2022-2026 ne prouve pas que
le modèle va bien performer en production — mais une régression claire
serait 